In [2]:
from datetime import datetime
import logging
import os

import ipydatetime
import ipywidgets as widgets

from getpass import getpass

import openstack
from keystoneclient.exceptions import NotFound
from taynacclient import client as taynacclient

from jinja2 import Environment, BaseLoader, FileSystemLoader, select_autoescape, StrictUndefined, Template

from IPython.display import display, HTML

In [3]:
# Set up logging - uncomment which level is appropriate
#LOG_LEVEL = logging.WARNING
LOG_LEVEL = logging.INFO
#LOG_LEVEL = logging.DEBUG

logging.basicConfig(format='%(message)s')
LOG = logging.getLogger(__name__)
LOG.setLevel(LOG_LEVEL)

# OpenStack auth settings

You'll want to use your admin credentials here.

`echo $OS_TOKEN`

and enter the result into the token field below.

In [4]:
# OpenStack auth settings
os.environ['OS_AUTH_URL'] = 'https://identity.rctest.nectar.org.au/'
os.environ['OS_AUTH_TYPE'] = 'token'
os.environ['OS_TOKEN'] = getpass(prompt='Enter your OpenStack Token')

In [ ]:
# Connect to OpenStack and confirm our token is good
conn = openstack.connect(cloud='envvars')
auth_ref = conn.session.auth.get_auth_ref(conn.session)
LOG.info(f"Token expires at: {auth_ref.expires}")

Token expires at: 2026-02-20 06:59:50+00:00


In [6]:
# Instantiate any clients we need
taynac = taynacclient.Client(version='1', session=conn.session)

In [12]:
# Set up the template environment
template_env = Environment(
    loader=FileSystemLoader('templates'),
    trim_blocks=True,
    undefined=StrictUndefined,
)

In [128]:
template_env = Environment(
    loader=BaseLoader(),
    undefined=StrictUndefined,
)

In [7]:
start_time = widgets.NaiveDatetimePicker(description='Start Time')
end_time = widgets.NaiveDatetimePicker(description='End Time')

timezone = widgets.Dropdown(
    options=[
        "Australia/Melbourne",
        "Australia/Adelaide",
        "Australia/Brisbane",
        "Australia/Hobart",
        "Australia/Perth",
        "Australia/Sydney",
        "Pacific/Auckland",
    ],
    description='Timezone')

def process(start_time, end_time, timezone):
    args = {
        "start_time": start_time,
        "end_time": end_time,        
        "timezone": timezone,
    }

out = widgets.interactive_output(
    process,
    {
        'start_time': start_time,
        'end_time': end_time,        
        'timezone': timezone,
    }
)

ui = widgets.HBox([
    widgets.VBox([
        start_time,
        end_time,
        timezone
    ]),
    out
])

display(ui)

In [12]:
def render_subject_string(subject, context):
    t = template_env.from_string(subject)
    return t.render(context).strip()

def render_body_string(body, context):
    t = template_env.from_string(body)
    return t.render(context).strip()
    
def render_body_template(filename, context):
    t = template_env.get_template(filename)
    return t.render(context).strip()

In [11]:
def send_notification(notification):
    taynac.messages.send(
        subject=notification['subject'],
        body=notification['body'],
        recipient=notification['to'],
        cc=notification['cc'],
    )

In [13]:
# Identity helper functions
def get_role(role_name):
    """Fetch project via the cache"""
    LOG.debug(f"Looking for role: {role_name}")
    return next(conn.identity.roles(name=role_name))

def get_project_users(project_id, role_id, exclude_disabled=False):
    """Get email addresses for users with certain roles
    in a given project."""
    LOG.debug(f"Getting users for project: {project_id} and role: {role_id}")
    users = []

    ras = conn.identity.role_assignments(scope_project_id=project_id, role_id=role_id, include_names=True)
    for ra in ras:
        u = get_user(ra.user.get('id'))
        if exclude_disabled and not u.enabled:
            continue
        # Only include users with an email address
        if getattr(u, 'email', None):
            users.append(u)
    return users

def get_project_members(project_id, exclude_disabled=False):
    users = []
    role_id = get_role('Member').get('id')
    return get_project_users(project_id, role_id, exclude_disabled=exclude_disabled)
                
def get_tenant_managers(project_id):
    """Get tenant manager emails for an instance."""
    role_id = get_role('TenantManager').get('id')
    return get_project_users(project_id, role_id)

def get_project(name_or_id):
    LOG.debug(f"Looking for project: {name_or_id}")
    return conn.identity.find_project(name_or_id, ignore_missing=False)

def get_user(name_or_id):
    LOG.debug(f"Looking for user: {name_or_id}")
    return conn.identity.find_user(name_or_id, ignore_missing=False)

def get_user1(name_or_id):
    """Fetch user"""
    user = None
    LOG.debug(f"Looking for user: {name_or_id}")
    try:
        user = conn.identity.users(id=name_or_id)
    except NotFound:
        user = conn.identity.users(name=name_or_id)
    finally:
        LOG.warning(f"Unknown User {name_or_id}")
    return user

In [16]:
def build_recipients(managers, members):
    """Generate list of recipients email addresses

    Returns one email address as the 'to' address which would generally
    be the first Tenant Manager of the project. Every other manager or
    member will be added to the 'cc' list.
    """
    combined = (managers or []) + (members or [])
    seen = set()
    emails = []
    for user in combined:
        email = getattr(user, "email", None)
        if email and email not in seen:
            seen.add(email)
            emails.append(email)
    if not emails:
        return None, []
    return emails[0], emails[1:]

In [14]:
def populate_data(instances):
    """Populate data from instances

    This function will build a dictionary of project, user and instance
    information from a given list of instances in a dict using the
    project ID as the key
    """
    data = {}  
    for instance in instances:
        project_id = instance['project_id']
        if not project_id in data:
            data[project_id] = {
                'project': get_project(project_id),
                'managers': get_tenant_managers(project_id),
                'members': get_project_members(project_id),
                'instances': [],
            }
        data[project_id]['instances'].append(instance)
    return data

In [26]:
subject = 'Test notification for project {{ project.name }}'

In [24]:
instances = list(conn.compute.servers(all_projects=True))
LOG.info(f"Found {len(instances)} instances")

Found 54 instances


In [30]:
data = populate_data(instances)

In [129]:
# Loop through the projects to generate the notifications
notifications = []
for d in list(data.values()):
    context = d.copy()
    context['start_ts'] = start_time.value
    context['end_ts'] = end_time.value
    context['tz'] = timezone.value

    # Calculate the duration
    duration = end_time.value - start_time.value
    context['days'] = duration.days
    context['hours'] = duration.seconds // 3600

    to_email, cc_emails = build_recipients(d['managers'], d['members'])

    rendered_subject = render_subject_string()
    rendered_body = render_body_template('host-planned-outage-notification.tmpl')
    
    notifications.append({
        'to': to_email,
        'cc': cc_emails,
        'subject': rendered_subject,
        'body': rendered_body,
    })

In [130]:
n = notifications[5]

display(n['to'])
display(n['cc'])
display(HTML(n['subject']))
display(HTML(n['body']))

'andy.botting@ardc.edu.au'

['s.crawley@uq.edu.au',
 'sam.morrison@unimelb.edu.au',
 'jake.yip@unimelb.edu.au']

In [131]:
instances = data['0b5c0bc217e4495e8d074ca5c61578b4']['instances']

In [132]:
instance = instances[1]

In [133]:
[','.join(a['addr']) for a in instance.addresses]

[]

In [134]:
instance

openstack.compute.v2.server.Server(id=ae5883c1-da66-48ad-b4c7-87b5401bd9e6, name=Ubuntu 24.04 LTS (Noble) Virtual Desktop, status=ERROR, tenant_id=0b5c0bc217e4495e8d074ca5c61578b4, user_id=f4eef31026dd483da97dea090ec9606b, metadata={}, hostId=, image={'id': 'd05c0835-fc43-4ae6-8867-79f1cf321d75', 'links': [{'rel': 'bookmark', 'href': 'https://compute.rctest.nectar.org.au/images/d05c0835-fc43-4ae6-8867-79f1cf321d75'}]}, flavor={'vcpus': 2, 'ram': 4096, 'disk': 30, 'ephemeral': 0, 'swap': 0, 'original_name': 'm3.small', 'extra_specs': {'nectar:rate': '0.057', 'production': 'true'}}, created=2025-12-17T04:01:50Z, updated=2025-12-17T04:02:50Z, addresses={}, accessIPv4=, accessIPv6=, links=[{'rel': 'self', 'href': 'https://compute.rctest.nectar.org.au/v2.1/servers/ae5883c1-da66-48ad-b4c7-87b5401bd9e6'}, {'rel': 'bookmark', 'href': 'https://compute.rctest.nectar.org.au/servers/ae5883c1-da66-48ad-b4c7-87b5401bd9e6'}], OS-DCF:diskConfig=MANUAL, OS-EXT-AZ:availability_zone=, config_drive=, key_

In [123]:
all_ips = [ip['addr'] for network in instance.addresses.values() for ip in network]
print(", ".join(all_ips))

In [91]:
for project in data.values():
    print(project['project']['name'], project['project']['id'])

tempest-melbourne 3cc2ce21208b4199930a49424c457747
test3 689a92eee01b4d20a5f93efe3daa6ad2
tempest-operator 62dcc07921664a6680c8aefcd17cd332
unimelb-devops-test ee7cc086d3464e59bd07c9db4b866bef
trove 1c9c222a3b2b4f0ebed7985c6242470c
bumblebee 0b5c0bc217e4495e8d074ca5c61578b4
tempest-monash 74e123582b1e42368496ee837210630b
octavia 238415faed3e4be1b709f6e6d1b6522a
pt-1256 01da1e72c6e04d0b9c889017ff94b22c
pt-1253 1b29db61f94147858f1edc991f6a3900
aucklandtest-demo a9ff6b5bbe0a4911b62660ad2fd471a3
unimelb-devops-test-2022 e12e607e285b4ff39d646e3f292b4722
admin e4eee8dbc16a49dcbc76edac96674e96
jupyterhub fc65462bc7f449d589713eaede979a64
rcmon_test 49412d513df74702bb02bfae8b79925f
tempest-auckland 82143e04a2f64334ac9279f45405725e
murano c4b36991a5f049ab8db1e062564954af
NeCTAR-Admins ad23531b159a4134aa5465b3099b6762
unimelb-test-sv-2024Mar12 7d38d461bc024bb3970e1e3cfb697675
unimelb-test-provisioner a6311f8e5c694c979af5d6523455f32e
monitoring 51804e8a3bec494a8266d1af96ebc019
